# `ptof_obs_nightly_baseline`

## What this notebook does
Recomputes the statistical baseline that detection depends on to tell "normal" from
"anomalous": per-capability, per-field response presence rate, and (added 2026-09-18)
write-lag and ETL-duration baselines. Nothing in this notebook detects anything itself — it
produces the reference points `ptof_obs_mal_output` and `ptof_obs_liveness_detection` compare
live data against.

## Position in the pipeline
- **Separate scheduled job** (`obs_nightly_baseline`) — runs nightly, independently of
  `obs_fresh_scan`. Not one of `obs_fresh_scan`'s tasks.
- **Upstream:** reads `v_llm_bronze`, `v_etl_bronze` (built by `ptof_obs_bronze_projection`) and
  `capability_registry` (human-curated by `ptof_obs_setup_seed`).
- **Downstream:** `ptof_obs_mal_output` reads `response_field_baseline` to detect schema drift
  (a field that used to reliably appear has gone missing). `ptof_obs_liveness_detection` reads
  `write_lag_baseline` and `etl_duration_baseline` for `write_lag_anomalies` / `etl_run_slow`.

## Why baselines are computed nightly, separately from detection
The queries do a `CREATE OR REPLACE` over a 30-day rolling window — expensive relative to the
hourly detection queries. Splitting this into its own nightly job means detection runs stay cheap
and compare against a stable reference point.

## Tables/views touched
- **Reads:** `v_llm_bronze`, `v_etl_bronze`, `capability_registry`
- **Writes:** `response_field_baseline` (per-capability, per-field presence rate),
  `write_lag_baseline`, `etl_duration_baseline`

## Write-lag / ETL-duration baselines (added 2026-09-18, provisional)
- `write_lag_baseline` — median + MAD of `v_llm_bronze.write_lag_s` per capability x
  scheduler_run (30-day window, floor n>=50, falls back to capability-only). Feeds
  `write_lag_anomalies` in `ptof_obs_liveness_detection` (WARN — log-only, does not notify).
- `etl_duration_baseline` — median + MAD of `v_etl_bronze.duration_seconds` per table_or_view x
  task_name (30-day window, `status='success'` only, floor n>=50, no fallback tier). Feeds
  `etl_run_slow` in `ptof_obs_liveness_detection` (WARN — log-only, does not notify).
- Both are pipeline write-lag / ETL-run-duration signals, not model-inference latency — see
  `threshold_basis` for the `'provisional'` status and the note that true inference latency
  (`latency_ms` on `ptof_primary__ai_llm_audit_log`) is blocked: that table has 0 rows in prod.

## Dropped baselines (prod migration 2026-09-10)
- `capability_latency_baseline` — no `latency_ms` in prod. Phase 2 via dev enrichment.

In [ ]:
%sql
-- response_field_baseline: per-capability, per-field presence rate computed nightly over a
-- 30-day rolling window. Supports ptof_obs_mal_output's response_schema_drift detector.
-- Prod context: all rows in ai_shift_outputs are successful outputs (no success/error columns),
-- so no success or credential-fastfail filter is needed. response_parsed is aliased from
-- content in the view, which is 100% valid JSON in prod.
-- Floor of 20 eligible rows per capability: prevents thin data from creating a noisy baseline.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.response_field_baseline AS
WITH eligible AS (
    -- non-blank outputs from active capabilities in the last 30 days
    SELECT b.capability, b.response_parsed
    FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
    JOIN mq_gmdf_dev.oil_obs.capability_registry r
      ON r.capability = b.capability AND r.active = true
    WHERE b.called_at >= current_timestamp() - INTERVAL 30 DAYS
      AND b.is_blank_output = false
),
row_counts AS (
    -- per-capability row counts, filtered to >= 20 to avoid thin baselines
    SELECT capability, count(*) AS n_rows FROM eligible GROUP BY capability HAVING count(*) >= 20
),
field_counts AS (
    -- explode each output's JSON keys and count how many rows each field appears in
    SELECT e.capability, k.key AS field_name, count(*) AS baseline_present
    FROM eligible e
    LATERAL VIEW explode(from_json(cast(e.response_parsed AS STRING), 'map<string,string>')) k AS key, val
    WHERE e.capability IN (SELECT capability FROM row_counts)
    GROUP BY e.capability, k.key
)
SELECT f.capability, f.field_name, f.baseline_present,
       r.n_rows AS baseline_total,
       f.baseline_present * 1.0 / r.n_rows AS baseline_presence_rate,
       current_timestamp() AS computed_at
FROM field_counts f
JOIN row_counts r ON r.capability = f.capability;

In [ ]:
%sql
-- write_lag_baseline: robust (median + MAD) baseline of v_llm_bronze.write_lag_s, per
-- capability x scheduler_run, over a 30-day rolling window. Feeds ptof_obs_liveness_detection's
-- write_lag_anomalies (WARN, provisional -- see threshold_basis). MAD instead of mean/stddev
-- because write_lag_s is right-skewed (a few slow writes, many fast ones); mean/stddev gets
-- dragged by the tail and inflates the threshold.
--
-- Floor of 50 rows per (capability, scheduler_run) stratum -- finer than that and the MAD is
-- computed off too few points to mean anything. Falls back to a capability-only baseline (same
-- 50-row floor) when the fine stratum doesn't qualify, so low-volume scheduler_run values (e.g.
-- ad-hoc reruns) still get a usable bound instead of no baseline at all.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.write_lag_baseline AS
WITH eligible AS (
    SELECT capability, scheduler_run, write_lag_s
    FROM mq_gmdf_dev.oil_obs.v_llm_bronze
    WHERE called_at >= current_timestamp() - INTERVAL 30 DAYS
      AND is_blank_output = false
),
fine_med AS (
    SELECT capability, scheduler_run,
           count(*)                              AS n,
           percentile_approx(write_lag_s, 0.5)    AS med_s
    FROM eligible
    GROUP BY capability, scheduler_run
),
fine AS (
    SELECT e.capability, e.scheduler_run, f.n, f.med_s,
           percentile_approx(abs(e.write_lag_s - f.med_s), 0.5) AS mad_s
    FROM eligible e
    JOIN fine_med f USING (capability, scheduler_run)
    GROUP BY e.capability, e.scheduler_run, f.n, f.med_s
    HAVING f.n >= 50
),
coarse_med AS (
    SELECT capability, count(*) AS n, percentile_approx(write_lag_s, 0.5) AS med_s
    FROM eligible
    GROUP BY capability
),
coarse AS (
    SELECT e.capability, c.n, c.med_s,
           percentile_approx(abs(e.write_lag_s - c.med_s), 0.5) AS mad_s
    FROM eligible e
    JOIN coarse_med c USING (capability)
    GROUP BY e.capability, c.n, c.med_s
    HAVING c.n >= 50
),
strata AS (
    SELECT DISTINCT capability, scheduler_run FROM eligible
)
SELECT
    s.capability,
    s.scheduler_run,
    CASE WHEN f.n IS NOT NULL THEN 'capability_scheduler' ELSE 'capability_only' END
        AS baseline_granularity,
    coalesce(f.n, c.n)                                      AS n,
    coalesce(f.med_s, c.med_s)                              AS median_write_lag_s,
    coalesce(f.mad_s, c.mad_s)                              AS mad_write_lag_s,
    coalesce(f.med_s, c.med_s) + 5 * 1.4826 * coalesce(f.mad_s, c.mad_s)
        AS upper_bound_s,
    current_timestamp()                                     AS computed_at
FROM strata s
LEFT JOIN fine f USING (capability, scheduler_run)
LEFT JOIN coarse c ON c.capability = s.capability
WHERE f.n IS NOT NULL OR c.n IS NOT NULL;

In [ ]:
%sql
-- etl_duration_baseline: robust (median + MAD) baseline of v_etl_bronze.duration_seconds, per
-- table_or_view x task_name, over a 30-day rolling window. Feeds ptof_obs_liveness_detection's
-- etl_run_slow (WARN, provisional -- see threshold_basis). Filtered to status='success' only --
-- a failed run's duration isn't a meaningful "how long does this normally take" data point (it
-- may have errored out early or hung before failing), and failures are already covered by the
-- separate etl_pipeline_health detector.
-- Floor of 50 rows per (table_or_view, task_name) stratum, same rationale as write_lag_baseline.
-- No fallback tier here (unlike write_lag_baseline) -- table_or_view x task_name is already the
-- coarsest grouping that makes sense (durations aren't comparable across different tables/tasks),
-- so a stratum under 50 rows simply has no baseline yet rather than a misleading broader one.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.etl_duration_baseline AS
WITH eligible AS (
    SELECT table_or_view, task_name, duration_seconds
    FROM mq_gmdf_dev.oil_obs.v_etl_bronze
    WHERE run_timestamp >= current_timestamp() - INTERVAL 30 DAYS
      AND status = 'success'
),
med AS (
    SELECT table_or_view, task_name,
           count(*)                                     AS n,
           percentile_approx(duration_seconds, 0.5)      AS med_s
    FROM eligible
    GROUP BY table_or_view, task_name
)
SELECT
    e.table_or_view,
    e.task_name,
    m.n,
    m.med_s                                                        AS median_duration_s,
    percentile_approx(abs(e.duration_seconds - m.med_s), 0.5)      AS mad_duration_s,
    m.med_s + 5 * 1.4826 * percentile_approx(abs(e.duration_seconds - m.med_s), 0.5)
                                                                    AS upper_bound_s,
    current_timestamp()                                            AS computed_at
FROM eligible e
JOIN med m USING (table_or_view, task_name)
GROUP BY e.table_or_view, e.task_name, m.n, m.med_s
HAVING m.n >= 50;